# wandb-finish — worked example 1: Close a wandb run after training for N epochs

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-finish`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Every `wandb.init()` call must be paired with a `wandb.finish()` call. Without `finish()`, the run remains open in wandb's buffer, metrics may not be fully uploaded, and calling `wandb.init()` again (e.g., in the next sweep trial) will either start a new run inside the existing one or raise a warning. Calling `finish()` cleanly closes the run and resets wandb's global state.

## Worked solution

**Step 1 — open the run.**
We call `wandb.init(project=..., name=...)` at the start of training. This opens a run and sets `wandb.run` to the active run handle. In production, this creates a real cloud run; here we mock wandb so no network call occurs.

**Step 2 — run the training loop.**
We loop `for epoch in range(n_epochs)`. The loop body does nothing here — the drill focuses on lifecycle, not content.

**Step 3 — close the run with wandb.finish().**
After the loop, we call `wandb.finish()`. This flushes remaining metrics to wandb's servers, marks the run as finished, and clears `wandb.run`. If we were running a sweep, this call is what tells the sweep controller to start the next trial.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def train_n_epochs(project, name, n_epochs):
    """Run a fake training loop with correct wandb lifecycle."""
    # Open the run
    wandb.init(project=project, name=name)
    
    # Training loop (fake)
    for epoch in range(n_epochs):
        pass  # real code would do forward/backward here
    
    # Close the run — always needed after init
    wandb.finish()
    return n_epochs

# Exercise it
result = train_n_epochs('my-project', 'exp-001', 5)
print('Epochs completed:', result)
print('wandb.init called:', wandb.init.called)
print('wandb.finish called:', wandb.finish.called)